In [0]:
dbutils.widgets.text("ENTITY", "orders", "Entity to process")
ENTITY = dbutils.widgets.get("ENTITY")
print(f"ENTITY from widget: {ENTITY}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import uuid


# ============================================================
# 1. ENTITY (read from widget parameter set by job task)
# ============================================================

# ENTITY is set in the first cell via dbutils.widgets.get("ENTITY")
# Job tasks pass base_parameters: {"ENTITY": "<entity_name>"}
#
# Supported entities:
# products
# orders
# order_items
# payments
# support_tickets


# ============================================================
# 2. ENTITY CONFIGURATION
# ============================================================

CONFIG = {

    # --------------------------------------------------------
    # PRODUCTS
    # --------------------------------------------------------

    "products": {
        "key": "product_id",

        "source_system": "PRODUCT_CATALOG",

        "bronze_table":
            "customer360_dev.bronze.products",

        "silver_table":
            "customer360_dev.silver.products",

        "work_table":
            "customer360_dev.control.product_cdc_work",

        "source_path":
            "/Volumes/customer360_dev/staging/"
            "source_files/products/incremental/",

        "columns": [
            "product_id",
            "product_name",
            "category_code",
            "subcategory_code",
            "brand_name",
            "unit_price",
            "cost_price",
            "product_status",
            "launch_date",
            "discontinued_date",
            "supplier_id",
            "currency_code",
            "created_at",
            "updated_at"
        ],

        "compare_columns": [
            "product_name",
            "category_code",
            "subcategory_code",
            "brand_name",
            "unit_price",
            "cost_price",
            "product_status",
            "launch_date",
            "discontinued_date",
            "supplier_id",
            "currency_code"
        ]
    },


    # --------------------------------------------------------
    # ORDERS
    # --------------------------------------------------------

    "orders": {
        "key": "order_id",

        "source_system": "ORDER_MANAGEMENT",

        "bronze_table":
            "customer360_dev.bronze.orders",

        "silver_table":
            "customer360_dev.silver.orders",

        "work_table":
            "customer360_dev.control.order_cdc_work",

        "source_path":
            "/Volumes/customer360_dev/staging/"
            "source_files/orders/incremental/",

        "columns": [
            "order_id",
            "customer_id",
            "order_timestamp",
            "order_status",
            "currency_code",
            "gross_amount",
            "discount_amount",
            "tax_amount",
            "shipping_amount",
            "net_amount",
            "shipping_address_id",
            "billing_address_id",
            "promotion_code",
            "created_at",
            "updated_at"
        ],

        "compare_columns": [
            "customer_id",
            "order_timestamp",
            "order_status",
            "currency_code",
            "gross_amount",
            "discount_amount",
            "tax_amount",
            "shipping_amount",
            "net_amount",
            "shipping_address_id",
            "billing_address_id",
            "promotion_code"
        ]
    },


    # --------------------------------------------------------
    # ORDER ITEMS
    # --------------------------------------------------------

    "order_items": {
        "key": "order_item_id",

        "source_system": "ORDER_MANAGEMENT",

        "bronze_table":
            "customer360_dev.bronze.order_items",

        "silver_table":
            "customer360_dev.silver.order_items",

        "work_table":
            "customer360_dev.control.order_item_cdc_work",

        "source_path":
            "/Volumes/customer360_dev/staging/"
            "source_files/order_items/incremental/",

        "columns": [
            "order_item_id",
            "order_id",
            "product_id",
            "quantity",
            "unit_price",
            "discount_amount",
            "tax_amount",
            "line_amount",
            "item_status",
            "created_at",
            "updated_at"
        ],

        "compare_columns": [
            "order_id",
            "product_id",
            "quantity",
            "unit_price",
            "discount_amount",
            "tax_amount",
            "line_amount",
            "item_status"
        ]
    },


    # --------------------------------------------------------
    # PAYMENTS
    # --------------------------------------------------------

    "payments": {
        "key": "payment_id",

        "source_system": "PAYMENT_PLATFORM",

        "bronze_table":
            "customer360_dev.bronze.payments",

        "silver_table":
            "customer360_dev.silver.payments",

        "work_table":
            "customer360_dev.control.payment_cdc_work",

        "source_path":
            "/Volumes/customer360_dev/staging/"
            "source_files/payments/incremental/",

        "columns": [
            "payment_id",
            "order_id",
            "customer_id",
            "payment_attempt_no",
            "payment_method",
            "payment_status",
            "payment_amount",
            "transaction_reference",
            "failure_reason",
            "payment_timestamp",
            "refund_amount",
            "refund_timestamp",
            "created_at",
            "updated_at"
        ],

        "compare_columns": [
            "order_id",
            "customer_id",
            "payment_attempt_no",
            "payment_method",
            "payment_status",
            "payment_amount",
            "transaction_reference",
            "failure_reason",
            "payment_timestamp",
            "refund_amount",
            "refund_timestamp"
        ]
    },


    # --------------------------------------------------------
    # SUPPORT TICKETS
    # --------------------------------------------------------

    "support_tickets": {
        "key": "ticket_id",

        "source_system": "SUPPORT_PLATFORM",

        "bronze_table":
            "customer360_dev.bronze.support_tickets",

        "silver_table":
            "customer360_dev.silver.support_tickets",

        "work_table":
            "customer360_dev.control.support_ticket_cdc_work",

        "source_path":
            "/Volumes/customer360_dev/staging/"
            "source_files/support_tickets/incremental/",

        "columns": [
            "ticket_id",
            "customer_id",
            "order_id",
            "ticket_category",
            "priority",
            "ticket_status",
            "channel",
            "agent_id",
            "created_timestamp",
            "first_response_timestamp",
            "resolved_timestamp",
            "closed_timestamp",
            "csat_score",
            "resolution_code",
            "created_at",
            "updated_at"
        ],

        "compare_columns": [
            "customer_id",
            "order_id",
            "ticket_category",
            "priority",
            "ticket_status",
            "channel",
            "agent_id",
            "created_timestamp",
            "first_response_timestamp",
            "resolved_timestamp",
            "closed_timestamp",
            "csat_score",
            "resolution_code"
        ]
    }
}


cfg = CONFIG[ENTITY]

KEY = cfg["key"]

print("Processing Entity:", ENTITY)
print("Business Key:", KEY)


# ============================================================
# 3. ENSURE _pipeline_run_id EXISTS
# ============================================================

bronze_columns = spark.table(
    cfg["bronze_table"]
).columns

if "_pipeline_run_id" not in bronze_columns:

    spark.sql(
        f"""
        ALTER TABLE {cfg['bronze_table']}
        ADD COLUMNS (
            _pipeline_run_id STRING
        )
        """
    )



    print(
        "_pipeline_run_id added to Bronze."
    )

else:

    print(
        "_pipeline_run_id already exists."
    )


# ============================================================
# 4. BUILD EXPLICIT STRING SCHEMA
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    DateType
)

source_schema = StructType(
    [
        StructField(
            column_name,
            StringType(),
            True
        )
        for column_name in cfg["columns"]
    ]
)


# ============================================================
# 5. READ INCREMENTAL SOURCE
# ============================================================

incremental_df = (
    spark.read
    .option("header", True)
    .schema(source_schema)
    .csv(
        cfg["source_path"]
    )
)


# ============================================================
# 6. CREATE PIPELINE RUN ID
# ============================================================

run_id = str(uuid.uuid4())

print(
    "Pipeline Run ID:",
    run_id
)


# ============================================================
# 7. ADD BRONZE METADATA
# ============================================================

bronze_incremental_df = (
    incremental_df

    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )

    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )

    .withColumn(
        "_source_system",
        F.lit(
            cfg["source_system"]
        )
    )

    .withColumn(
        "_load_type",
        F.lit("INCREMENTAL")
    )

    .withColumn(
        "_pipeline_run_id",
        F.lit(run_id)
    )
)


# ============================================================
# 8. COUNT + BRONZE APPEND
# ============================================================

incoming_count = (
    bronze_incremental_df.count()
)

print(
    f"Incoming {ENTITY} rows:",
    incoming_count
)

if incoming_count > 0:

    (
        bronze_incremental_df
        .write
        .format("delta")
        .mode("append")
        .option(
            "mergeSchema",
            "true"
        )
        .saveAsTable(
            cfg["bronze_table"]
        )
    )

    print(
        "Bronze incremental load completed."
    )

else:

    print(
        "No incremental rows found. "
        "Bronze write skipped."
    )


# ============================================================
# 9. STOP CDC LOGIC IF NO DATA
# ============================================================

if incoming_count == 0:

    print(
        f"No incremental {ENTITY} data. "
        "CDC processing ends successfully."
    )

else:

    # ========================================================
    # 10. READ CURRENT RUN FROM BRONZE
    # ========================================================

    current_bronze_df = (
        spark.table(
            cfg["bronze_table"]
        )
        .filter(
            F.col("_pipeline_run_id")
            ==
            run_id
        )
    )


    # ========================================================
    # 11. LOAD SILVER
    # ========================================================

    silver_df = spark.table(
        cfg["silver_table"]
    )


    # ========================================================
    # 12. DEDUP CURRENT RUN
    # ========================================================

    current_bronze_df = (
        current_bronze_df
        .withColumn(
            "_updated_at_cast",
            F.coalesce(
                F.try_to_timestamp("updated_at"),
                F.expr("try_to_timestamp(updated_at, 'dd-MM-yyyy HH:mm')"),
                F.expr("try_to_timestamp(updated_at, 'dd-MM-yyyy')")
            )
        )
    )

    entity_window = (
        Window
        .partitionBy(KEY)
        .orderBy(
            F.col(
                "_updated_at_cast"
            ).desc_nulls_last(),

            F.col(
                "_ingestion_timestamp"
            ).desc()
        )
    )

    dedup_df = (
        current_bronze_df
        .withColumn(
            "_row_number",
            F.row_number().over(
                entity_window
            )
        )
        .filter(
            F.col("_row_number") == 1
        )
        .drop("_row_number")
    )


    # ========================================================
    # 13. CAST SOURCE TO SILVER DATA TYPES
    # ========================================================

    silver_schema = {
        field.name: field.dataType
        for field
        in silver_df.schema.fields
    }

    casted_df = dedup_df

    for column_name in cfg[
        "compare_columns"
    ] + [
        "created_at",
        "updated_at"
    ]:

        if (
            column_name
            in casted_df.columns
            and
            column_name
            in silver_schema
        ):

            target_type = silver_schema[column_name]

            if isinstance(target_type, TimestampType):
                casted_df = (
                    casted_df
                    .withColumn(
                        column_name,
                        F.coalesce(
                            F.try_to_timestamp(F.col(column_name)),
                            F.expr(f"try_to_timestamp({column_name}, 'dd-MM-yyyy HH:mm')"),
                            F.expr(f"try_to_timestamp({column_name}, 'dd-MM-yyyy')")
                        )
                    )
                )
            elif isinstance(target_type, DateType):
                casted_df = (
                    casted_df
                    .withColumn(
                        column_name,
                        F.coalesce(
                            F.try_to_timestamp(F.col(column_name)),
                            F.expr(f"try_to_timestamp({column_name}, 'dd-MM-yyyy HH:mm')"),
                            F.expr(f"try_to_timestamp({column_name}, 'dd-MM-yyyy')")
                        ).cast("date")
                    )
                )
            else:
                casted_df = (
                    casted_df
                    .withColumn(
                        column_name,
                        F.col(column_name).cast(target_type)
                    )
                )


    # ========================================================
    # 14. JOIN SOURCE TO SILVER
    # ========================================================

    comparison_df = (
        casted_df.alias("src")
        .join(
            silver_df.alias("tgt"),

            F.col(
                f"src.{KEY}"
            )
            ==
            F.col(
                f"tgt.{KEY}"
            ),

            how="left"
        )
    )


    # ========================================================
    # 15. BUILD CHANGE CONDITION
    # ========================================================

    change_condition = None

    for column_name in cfg[
        "compare_columns"
    ]:

        condition = (
            ~F.col(
                f"src.{column_name}"
            ).eqNullSafe(
                F.col(
                    f"tgt.{column_name}"
                )
            )
        )

        if change_condition is None:

            change_condition = condition

        else:

            change_condition = (
                change_condition
                |
                condition
            )


    # ========================================================
    # 16. CLASSIFY CDC
    # ========================================================

    classified_df = (
        comparison_df
        .withColumn(
            "cdc_action",

            F.when(
                F.col(
                    f"tgt.{KEY}"
                ).isNull(),

                F.lit("INSERT")
            )

            .when(
                change_condition,
                F.lit("UPDATE")
            )

            .otherwise(
                F.lit("UNCHANGED")
            )
        )
    )


    # ========================================================
    # 17. BUILD CDC OUTPUT
    # ========================================================

    business_output_columns = []

    for column_name in cfg[
        "columns"
    ]:

        if column_name in casted_df.columns:

            business_output_columns.append(
                F.col(
                    f"src.{column_name}"
                ).alias(
                    column_name
                )
            )

    cdc_df = (
        classified_df
        .select(
            *business_output_columns,

            F.col(
                "src._source_file"
            ).alias(
                "_source_file"
            ),

            F.col(
                "src._source_system"
            ).alias(
                "_source_system"
            ),

            F.col(
                "src._load_type"
            ).alias(
                "_load_type"
            ),

            F.col(
                "src._ingestion_timestamp"
            ).alias(
                "_ingestion_timestamp"
            ),

            F.col(
                "src._pipeline_run_id"
            ).alias(
                "_pipeline_run_id"
            ),

            F.col(
                "cdc_action"
            )
        )
    )


    # ========================================================
    # 18. PERSIST CDC WORK TABLE
    # ========================================================

    (
        cdc_df
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(
            cfg["work_table"]
        )
    )


    # ========================================================
    # 19. CDC COUNTS
    # ========================================================

    insert_count = (
        cdc_df
        .filter(
            F.col("cdc_action")
            ==
            "INSERT"
        )
        .count()
    )

    update_count = (
        cdc_df
        .filter(
            F.col("cdc_action")
            ==
            "UPDATE"
        )
        .count()
    )

    unchanged_count = (
        cdc_df
        .filter(
            F.col("cdc_action")
            ==
            "UNCHANGED"
        )
        .count()
    )

    print(
        "INSERT:",
        insert_count
    )

    print(
        "UPDATE:",
        update_count
    )

    print(
        "UNCHANGED:",
        unchanged_count
    )


    # ========================================================
    # 20. PREPARE MERGE SOURCE
    # ========================================================

    merge_source_df = (
        cdc_df
        .filter(
            F.col(
                "cdc_action"
            ).isin(
                "INSERT",
                "UPDATE"
            )
        )
    )


    # ========================================================
    # 21. BUILD SILVER COLUMN MAPPING
    # ========================================================

    excluded_target_columns = {
        KEY,
        "_silver_processed_at"
    }

    merge_columns = [
        c
        for c
        in silver_df.columns
        if (
            c not in
            excluded_target_columns
            and
            c in merge_source_df.columns
        )
    ]

    update_mapping = {
        column_name:
            f"src.{column_name}"
        for column_name
        in merge_columns
    }

    if (
        "_silver_processed_at"
        in silver_df.columns
    ):

        update_mapping[
            "_silver_processed_at"
        ] = "current_timestamp()"


    insert_mapping = {
        KEY:
            f"src.{KEY}"
    }

    for column_name in merge_columns:

        insert_mapping[
            column_name
        ] = (
            f"src.{column_name}"
        )

    if (
        "_silver_processed_at"
        in silver_df.columns
    ):

        insert_mapping[
            "_silver_processed_at"
        ] = "current_timestamp()"


    # ========================================================
    # 22. SILVER COUNT BEFORE MERGE
    # ========================================================

    silver_before_count = (
        silver_df.count()
    )


    # ========================================================
    # 23. DELTA MERGE
    # ========================================================

    silver_delta = (
        DeltaTable.forName(
            spark,
            cfg["silver_table"]
        )
    )

    (
        silver_delta.alias("tgt")

        .merge(
            merge_source_df.alias("src"),

            f"""
            tgt.{KEY}
            =
            src.{KEY}
            """
        )

        .whenMatchedUpdate(
            condition="""
                src.cdc_action = 'UPDATE'
                AND
                (
                    tgt.updated_at IS NULL
                    OR
                    src.updated_at >=
                    tgt.updated_at
                )
            """,

            set=update_mapping
        )

        .whenNotMatchedInsert(
            condition="""
                src.cdc_action = 'INSERT'
            """,

            values=insert_mapping
        )

        .execute()
    )


    # ========================================================
    # 24. SILVER COUNT AFTER MERGE
    # ========================================================

    silver_after_df = spark.table(
        cfg["silver_table"]
    )

    silver_after_count = (
        silver_after_df.count()
    )

    actual_growth = (
        silver_after_count
        -
        silver_before_count
    )

    print(
        "Silver growth:",
        actual_growth
    )

    print(
        "Expected inserts:",
        insert_count
    )


    # ========================================================
    # 25. RECONCILIATION
    # ========================================================

    source_distinct_count = (
        current_bronze_df
        .select(KEY)
        .distinct()
        .count()
    )

    classified_count = (
        insert_count
        +
        update_count
        +
        unchanged_count
    )

    classification_difference = (
        source_distinct_count
        -
        classified_count
    )


    # ========================================================
    # 26. VERIFY INSERTS
    # ========================================================

    missing_insert_count = (
        cdc_df
        .filter(
            F.col("cdc_action")
            ==
            "INSERT"
        )
        .select(KEY)
        .join(
            silver_after_df.select(KEY),

            on=KEY,

            how="left_anti"
        )
        .count()
    )


    # ========================================================
    # 27. VERIFY UPDATES
    # ========================================================

    missing_update_count = (
        cdc_df
        .filter(
            F.col("cdc_action")
            ==
            "UPDATE"
        )
        .select(KEY)
        .join(
            silver_after_df.select(KEY),

            on=KEY,

            how="left_anti"
        )
        .count()
    )


    # ========================================================
    # 28. VERIFY KEY UNIQUENESS
    # ========================================================

    duplicate_key_count = (
        silver_after_df
        .groupBy(KEY)
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )


    # ========================================================
    # 29. FINAL PASS / FAIL
    # ========================================================

    reconciliation_pass = (
        classification_difference == 0
        and
        missing_insert_count == 0
        and
        missing_update_count == 0
        and
        duplicate_key_count == 0
        and
        actual_growth == insert_count
    )

    status = (
        "PASS"
        if reconciliation_pass
        else "FAIL"
    )

    print(
        f"{ENTITY} CDC reconciliation:",
        status
    )


    # ========================================================
    # 30. DISPLAY RESULT
    # ========================================================

    result = [
        {
            "entity":
                ENTITY,

            "pipeline_run_id":
                run_id,

            "source_distinct":
                source_distinct_count,

            "insert_count":
                insert_count,

            "update_count":
                update_count,

            "unchanged_count":
                unchanged_count,

            "classification_difference":
                classification_difference,

            "missing_inserts":
                missing_insert_count,

            "missing_updates":
                missing_update_count,

            "duplicate_keys":
                duplicate_key_count,

            "silver_growth":
                actual_growth,

            "status":
                status
        }
    ]

    display(
        spark.createDataFrame(
            result
        )
    )


    # ========================================================
    # 31. UPDATE WATERMARK ONLY AFTER PASS
    # ========================================================

    if status == "PASS":

        latest_processed = (
            cdc_df
            .agg(
                F.max(
                    "updated_at"
                ).alias(
                    "last_processed_value"
                ),

                F.max(
                    "_source_file"
                ).alias(
                    "last_processed_file"
                )
            )
            .first()
        )

        last_processed_value = (
            latest_processed[
                "last_processed_value"
            ]
        )

        last_processed_file = (
            latest_processed[
                "last_processed_file"
            ]
        )

        watermark_update_df = (
            spark.createDataFrame(
                [
                    (
                        ENTITY,
                        cfg[
                            "source_system"
                        ],
                        last_processed_value,
                        last_processed_file,
                        run_id,
                        "SUCCESS"
                    )
                ],

                """
                entity_name STRING,
                source_system STRING,
                last_processed_value TIMESTAMP,
                last_processed_file STRING,
                last_run_id STRING,
                load_status STRING
                """
            )

            .withColumn(
                "updated_at",
                F.current_timestamp()
            )
        )

        watermark_update_df.createOrReplaceTempView(
            "cdc_watermark_update"
        )

        spark.sql("""
            MERGE INTO
                customer360_dev.control.watermark
                AS tgt

            USING
                cdc_watermark_update
                AS src

            ON
                tgt.entity_name
                =
                src.entity_name

            WHEN MATCHED THEN
            UPDATE SET

                tgt.source_system =
                    src.source_system,

                tgt.last_processed_value =
                    src.last_processed_value,

                tgt.last_processed_file =
                    src.last_processed_file,

                tgt.last_run_id =
                    src.last_run_id,

                tgt.load_status =
                    src.load_status,

                tgt.updated_at =
                    src.updated_at

            WHEN NOT MATCHED THEN

            INSERT (
                entity_name,
                source_system,
                last_processed_value,
                last_processed_file,
                last_run_id,
                load_status,
                updated_at
            )

            VALUES (
                src.entity_name,
                src.source_system,
                src.last_processed_value,
                src.last_processed_file,
                src.last_run_id,
                src.load_status,
                src.updated_at
            )
        """)

        print(
            f"{ENTITY} watermark updated."
        )

    else:

        print(
            f"{ENTITY} reconciliation failed. "
            "Watermark NOT updated."
        )

Processing Entity: products
Business Key: product_id
_pipeline_run_id already exists.
Pipeline Run ID: 60ad30f7-2052-427d-b9d2-0f0fde50520a
Incoming products rows: 20
Bronze incremental load completed.
INSERT: 10
UPDATE: 0
UNCHANGED: 10
Silver growth: 10
Expected inserts: 10
products CDC reconciliation: PASS


classification_difference,duplicate_keys,entity,insert_count,missing_inserts,missing_updates,pipeline_run_id,silver_growth,source_distinct,status,unchanged_count,update_count
0,0,products,10,0,0,60ad30f7-2052-427d-b9d2-0f0fde50520a,10,20,PASS,10,0


products watermark updated.


In [0]:
# This cell previously overwrote ENTITY with hardcoded values.
# Removed — ENTITY now comes from the job task base_parameters via dbutils.widgets.get().